# LLM4Teach — Reflection + LLM Planner Experiments

**Purpose:** debugger · experiment runner · reflection validation suite · convergence analysis

This notebook imports repository modules directly. It does NOT redefine any training logic.

**Sections:**
1. Environment Setup & Connectivity Checks
2. Reflection Pipeline Validation
3. Strict Parser Validation
4. Planner LLM Experiments
5. Reflection LLM Experiments
6. Planner + Reflection Integration
7. PPO Training Experiments
8. Convergence Visualization
9. Reflection Memory Inspection
10. Ablation Experiments
11. Failure Case Debugging


## § 0  —  Repo root setup

In [ ]:
import sys, os

_this_dir = os.getcwd()
if os.path.basename(_this_dir) == 'notebooks':
    REPO_ROOT = os.path.abspath(os.path.join(_this_dir, '..'))
elif os.path.exists(os.path.join(_this_dir, 'Game.py')):
    REPO_ROOT = _this_dir
else:
    _d = _this_dir
    while _d != os.path.dirname(_d):
        if os.path.exists(os.path.join(_d, 'Game.py')):
            REPO_ROOT = _d
            break
        _d = os.path.dirname(_d)
    else:
        raise RuntimeError('Could not locate LLM4Teach repo root')

os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print('Repo root :', REPO_ROOT)
print('Python    :', sys.version)


## § 1  —  Environment Setup & Connectivity Checks

In [ ]:
import torch
import requests

# ── Configurable experiment parameters ───────────────────────────────────────
CFG = dict(
    task                 = 'SimpleDoorKey',
    n_itr                = 100,
    traj_per_itr         = 5,
    batch_size           = 64,
    reflection_maxlen    = 20,
    reflection_top_k     = 5,
    planner_model        = 'qwen2.5:3b',
    reflector_model      = 'qwen2.5:7b',
    planner_backend      = 'ollama',     # 'offline' | 'ollama' | 'dashscope'
    reflector_backend    = 'ollama',
    planner_temperature  = 0.2,
    reflect_temperature  = 0.4,
    ollama_url           = 'http://localhost:11434',
    device               = 'cuda' if torch.cuda.is_available() else 'cpu',
)

print('=== Experiment Configuration ===')
for k, v in CFG.items():
    print(f'  {k:25s} = {v}')
print()

print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')


In [ ]:
# ── Ollama connectivity check ─────────────────────────────────────────────────
print('=== Ollama Connectivity ===')
try:
    r = requests.get(f"{CFG['ollama_url']}/api/tags", timeout=5)
    if r.status_code == 200:
        models = [m['name'] for m in r.json().get('models', [])]
        print(f'Ollama reachable. Installed models: {models}')
        for needed in [CFG['planner_model'], CFG['reflector_model']]:
            base = needed.split(':')[0]
            found = any(base in m for m in models)
            status = '✅' if found else '❌ NOT PULLED'
            print(f'  {status}  {needed}')
            if not found:
                print(f'       → Run: ollama pull {needed}')
    else:
        print(f'Ollama returned HTTP {r.status_code}')
except Exception as e:
    print(f'Ollama NOT reachable: {e}')
    print('Start Ollama with: ollama serve')


In [ ]:
# ── Import all repository modules ─────────────────────────────────────────────
from Game import Game
from memory.reflection import QwenReflector, EpisodeTrajectory, validate_reflection
from memory.memory_buffer import ReflectionMemory
from utils.symbolic_parser import validate_plan, parse_and_validate, strict_parse, VALID_OBJECTS
from utils.qwen_llm import QwenLLM, robust_parse_plan
from planner import Planner
from teacher_policy import TeacherPolicy
import numpy as np

print('All repository imports OK.')
print(f'Valid symbolic objects ({len(VALID_OBJECTS)}):', sorted(VALID_OBJECTS))


In [ ]:
# ── Verify planner LLM inference end-to-end ───────────────────────────────────
print('=== Planner LLM Inference Verification ===')
qwen_planner = QwenLLM(
    backend     = CFG['planner_backend'],
    model       = CFG['planner_model'],
    temperature = CFG['planner_temperature'],
)
ok = qwen_planner.verify_inference()
print('Planner inference OK:', ok)


## § 2  —  Reflection Pipeline Validation

In [ ]:
# Tests for validate_reflection() — good and bad examples
TEST_REFLECTIONS = [
    # Expected VALID
    ('VALID',   'Exploring room boundaries early revealed the blue key faster. After obtaining the matching key, unnecessary exploration was reduced.'),
    ('VALID',   'The agent found the key quickly by scanning unexplored areas. Opening the door immediately after picking up the key was efficient.'),
    # Expected REJECTED — coordinate
    ('REJECT',  'The key was found at position (7,3). Then the door was at (4,2).'),
    ('REJECT',  'Try grid cell 5,3 to find the key faster.'),
    # Expected REJECTED — speculation / hidden state
    ('REJECT',  'There was probably a hidden room behind the wall.'),
    ('REJECT',  'The key is always in the lower room — a permanent world rule.'),
    # Expected REJECTED — fabricated objects
    ('REJECT',  'The bookshelf contained a key near the emergency exit.'),
    ('REJECT',  'Use the staircase to reach the door on the upper floor.'),
]

print('Reflection validation tests:')
print('-' * 70)
passed = 0
for expected, text in TEST_REFLECTIONS:
    ok = validate_reflection(text)
    actual = 'VALID' if ok else 'REJECT'
    match = '✅' if actual == expected else '❌ WRONG'
    print(f'{match} Expected={expected:6s} Got={actual:6s} | {text[:60]}')
    if actual == expected:
        passed += 1

print(f'\nPassed: {passed}/{len(TEST_REFLECTIONS)}')


## § 3  —  Strict Parser Validation

In [ ]:
TEST_PLANS = [
    # (expected_valid, plan_string)
    (True,  'explore'),
    (True,  'go to <key>, pick up <key>'),
    (True,  'go to <door>, open <door>'),
    (True,  'drop <key>, go to <red door>, open <red door>'),
    (True,  'go to <blue key>, pick up <blue key>, go to <blue door>, open <blue door>'),
    # Invalid — unknown object
    (False, 'go to <handle>, pick up <handle>'),
    (False, 'open <button>'),
    (False, 'go to <room>'),
    # Invalid — coordinates
    (False, 'go to (7,3)'),
    (False, 'move to position 5,3'),
    # Invalid — missing <> wrapper
    (False, 'go to key'),
    (False, 'pick up the blue thing'),
    # Invalid — free-form language
    (False, 'I think the agent should pick up the key first.'),
    # Empty
    (False, ''),
]

print('Strict parser validation tests:')
print('-' * 70)
passed = 0
for expected_valid, plan in TEST_PLANS:
    result = parse_and_validate(plan)
    actually_valid = result is not None
    ok, errs = validate_plan(plan)
    match = '✅' if actually_valid == expected_valid else '❌'
    clean = result if result else '(rejected)'
    print(f"{match} exp={'VALID' if expected_valid else 'REJECT':6s} | in='{plan[:40]:40s}' → '{clean[:40]}'")
    if actually_valid == expected_valid:
        passed += 1

print(f'\nPassed: {passed}/{len(TEST_PLANS)}')


In [ ]:
# Deduplication test
dup_plan = 'go to <key>, pick up <key>, go to <key>, open <door>, pick up <key>'
clean    = parse_and_validate(dup_plan)
print(f'Input : {dup_plan}')
print(f'Output: {clean}')
assert clean == 'go to <key>, pick up <key>, open <door>', f'Unexpected: {clean}'
print('Deduplication OK.')


## § 4  —  Planner LLM Experiments

In [ ]:
# Direct planner LLM test on representative MiniGrid observations
PLANNER_TEST_OBS = [
    'Agent sees <nothing>, holds <nothing>.',
    'Agent sees <key>, holds <nothing>.',
    'Agent sees <door>, holds <nothing>.',
    'Agent sees <door>, holds <key>.',
    'Agent sees <key>, <door>, holds <nothing>.',
    'Agent sees <blue key>, <blue door>, holds <nothing>.',
]

print('=== Single-shot planner test ===')
for obs in PLANNER_TEST_OBS:
    if CFG['planner_backend'] == 'offline':
        print(f'OBS: {obs}')
        print('  (offline backend — no LLM call)')
    else:
        plan, raw = qwen_planner.call_with_retry(user_prompt=obs)
        validated = parse_and_validate(plan) if plan else None
        print(f'OBS      : {obs}')
        print(f'Raw      : {(raw or "")[:80]}')
        print(f'Parsed   : {plan}')
        print(f'Validated: {validated}')
    print()


In [ ]:
# Consistency test: same observation, N repeated samples
N_SAMPLES = 10
test_obs = 'Agent sees <key>, <door>, holds <nothing>.'

outputs = []
if CFG['planner_backend'] != 'offline':
    for i in range(N_SAMPLES):
        plan, _ = qwen_planner.call_with_retry(user_prompt=test_obs)
        validated = parse_and_validate(plan) if plan else 'explore'
        outputs.append(validated)

    from collections import Counter
    counts = Counter(outputs)
    total = len(outputs)
    print(f'Consistency test ({N_SAMPLES} samples):')
    print(f'Observation: {test_obs}')
    for plan, cnt in counts.most_common():
        bar = '█' * int(cnt / total * 30)
        print(f'  {cnt/total:.0%} {bar} → {plan}')

    consistency = counts.most_common(1)[0][1] / total
    print(f'\nMost-common plan consistency: {consistency:.0%}')
    print('Planner stats:', qwen_planner.stats)
else:
    print('Offline backend — skip consistency test')


In [ ]:
# Planner latency benchmark
import time

if CFG['planner_backend'] != 'offline':
    N_BENCH = 5
    latencies = []
    for _ in range(N_BENCH):
        t0 = time.perf_counter()
        qwen_planner.call_with_retry(user_prompt='Agent sees <key>, holds <nothing>.')
        latencies.append(time.perf_counter() - t0)
    print(f'Planner latency ({N_BENCH} calls):')
    print(f'  mean  = {np.mean(latencies):.2f}s')
    print(f'  min   = {np.min(latencies):.2f}s')
    print(f'  max   = {np.max(latencies):.2f}s')
else:
    print('Offline backend — no latency measurement')


## § 5  —  Reflection LLM Experiments

In [ ]:
from memory.reflection import clean_reflection

reflector = QwenReflector(
    backend     = CFG['reflector_backend'],
    model       = CFG['reflector_model'],
    temperature = CFG['reflect_temperature'],
)

# Build synthetic trajectories of different types
def make_traj(obs_plan_reward_list, success):
    t = EpisodeTrajectory()
    for obs, plan, rew in obs_plan_reward_list:
        t.add_step(obs, plan, rew)
    t.finish(success=success)
    return t

TRAJ_CASES = [
    ('Success (direct)',
     make_traj([
         ('Agent sees <key>, holds <nothing>.',    'go to <key>, pick up <key>',   0.0),
         ('Agent sees <door>, holds <key>.',       'go to <door>, open <door>',    1.0),
     ], success=True)),
    ('Failure (exploration-heavy)',
     make_traj([
         ('Agent sees <nothing>, holds <nothing>.', 'explore', 0.0)] * 20 +
        [('Agent sees <key>, holds <nothing>.',    'go to <key>, pick up <key>', 0.0),
         ('Agent sees <nothing>, holds <key>.',    'explore',                    0.0)] * 10,
     success=False)),
]

print('=== Reflection Generation ===')
for label, traj in TRAJ_CASES:
    print(f'\n--- {label} ---')
    print(traj.to_prompt()[:400])
    print('\nGenerating reflection...')
    if CFG['reflector_backend'] != 'offline':
        reflection = reflector.reflect(traj)
        print(f'Reflection : {reflection}')
        print(f'Valid      : {validate_reflection(reflection) if reflection else "N/A"}')
    else:
        print('(offline backend — no reflection generated)')


In [ ]:
# Reflection latency benchmark
if CFG['reflector_backend'] != 'offline':
    traj_bench = TRAJ_CASES[0][1]
    N_BENCH = 3
    latencies = []
    for _ in range(N_BENCH):
        t0 = time.perf_counter()
        reflector.reflect(traj_bench)
        latencies.append(time.perf_counter() - t0)
    print(f'Reflector latency ({N_BENCH} calls):')
    print(f'  mean = {np.mean(latencies):.2f}s  min = {np.min(latencies):.2f}s  max = {np.max(latencies):.2f}s')
    print('Reflector stats:', reflector.stats)
else:
    print('Offline backend — no latency measurement')


## § 6  —  Planner + Reflection Integration

In [ ]:
# Build a game instance and wire memory into the planner
import argparse

args = argparse.Namespace(
    task='SimpleDoorKey', frame_stack=1,
    offline_planner=True, soft_planner=False,
    seed=42, seed_list=[42], n_itr=10, traj_per_itr=5,
    batch_size=64, gamma=0.99, lam=0.95, recurrent=False,
    device=CFG['device'],
    logdir=os.path.join(REPO_ROOT, 'log'), savedir='notebook-integration-42',
    loaddir=None, loadmodel='acmodel', policy='ppo',
    num_eval=5, eval_interval=5, save_interval=5, eval_teacher=False,
    env_seed_list=[0],
)

integ_game = Game(args)
integ_memory = ReflectionMemory(maxlen=10, top_k=3)

# Add a synthetic reflection
integ_memory.add_memory(
    'Exploring room edges revealed the key faster than random search.',
    episode_id=0, success=True, ep_len=40, total_reward=1.0
)

# Wire memory into planner
integ_game.teacher_policy.planner.set_reflection_memory(integ_memory)

print('Integration setup complete.')


In [ ]:
# Inspect prompt augmentation
obs = integ_game.env.reset()
obs_text = integ_game.teacher_policy.planner.mediator.RL2LLM(obs[0])

plain_prompt     = obs_text
augmented_prompt = integ_game.teacher_policy.planner._build_prompt_with_reflection(obs_text)

print('Observation text (cache key — UNCHANGED):')
print(' ', obs_text)
print()
print('Augmented prompt sent to LLM (reflection injected):')
print('-' * 60)
print(augmented_prompt)
print('-' * 60)
print()
print(f'Cache key  length : {len(plain_prompt)}')
print(f'Augmented  length : {len(augmented_prompt)}')
print(f'Reflection added  : {len(augmented_prompt) > len(plain_prompt)}')


In [ ]:
# Compare planner output WITH vs WITHOUT reflection (requires online LLM)
if CFG['planner_backend'] != 'offline':
    from utils.qwen_llm import MINIGRID_SYSTEM_PROMPT

    test_obs = 'Agent sees <key>, <door>, holds <nothing>.'

    # Without reflection
    plan_no_reflect, _ = qwen_planner.call_with_retry(user_prompt=test_obs)

    # With reflection injected
    context = integ_memory.get_context()
    augmented = f'{context}\n\nCurrent observation:\n{test_obs}'
    plan_with_reflect, _ = qwen_planner.call_with_retry(user_prompt=augmented)

    print('Comparison: WITH vs WITHOUT reflection')
    print(f'Without : {plan_no_reflect}')
    print(f'With    : {plan_with_reflect}')
    print(f'Same    : {plan_no_reflect == plan_with_reflect}')
else:
    print('Offline backend — comparison requires online LLM')


## § 7  —  PPO Training Experiments

In [ ]:
# Helper to build a Game with configurable settings
def make_game(savedir, n_itr=50, offline=True, reflection=False,
              reflector_backend='offline', seed=42):
    a = argparse.Namespace(
        task=CFG['task'], frame_stack=1,
        offline_planner=offline, soft_planner=False,
        seed=seed, seed_list=[seed], n_itr=n_itr, traj_per_itr=5,
        batch_size=64, gamma=0.99, lam=0.95, recurrent=False,
        device=CFG['device'],
        logdir=os.path.join(REPO_ROOT, 'log'), savedir=savedir,
        loaddir=None, loadmodel='acmodel', policy='ppo',
        num_eval=5, eval_interval=10, save_interval=25, eval_teacher=False,
        env_seed_list=[0],
    )
    g = Game(a)
    if reflection:
        refl = QwenReflector(backend=reflector_backend,
                             model=CFG['reflector_model'],
                             temperature=CFG['reflect_temperature'])
        mem  = ReflectionMemory(maxlen=CFG['reflection_maxlen'],
                                top_k=CFG['reflection_top_k'])
        g.set_reflection_system(refl, mem)
    return g


In [ ]:
# Condition 1: PPO only (offline planner, no reflection)
N_ITR = 50   # short run for notebook
print(f'Running PPO-only ({N_ITR} iterations)...')
game_ppo = make_game('exp-ppo-only-42', n_itr=N_ITR)
game_ppo.train()
print('PPO-only done. Log dir:', game_ppo.logger.dir)


In [ ]:
# Condition 2: PPO + symbolic planner (offline)
print(f'Running PPO + planner ({N_ITR} iterations)...')
game_planner = make_game('exp-ppo-planner-42', n_itr=N_ITR, offline=True)
game_planner.train()
print('PPO+planner done. Log dir:', game_planner.logger.dir)


In [ ]:
# Condition 3: PPO + planner + reflection (offline reflector → no LLM needed)
print(f'Running PPO + planner + reflection ({N_ITR} iterations)...')
game_reflect = make_game('exp-ppo-reflect-42', n_itr=N_ITR,
                         reflection=True, reflector_backend='offline')
game_reflect.train()
print('PPO+reflect done. Log dir:', game_reflect.logger.dir)


## § 8  —  Convergence Visualization

In [ ]:
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['figure.dpi'] = 120

from simulator.visualize_training import (
    load_tb_scalars, plot_reward_curve, plot_success_rate,
    plot_ppo_losses, plot_reflection_stats, _unzip, _smooth,
)


In [ ]:
# Multi-run comparison: reward curves
runs = {
    'PPO only':            game_ppo.logger.dir,
    'PPO + planner':       game_planner.logger.dir,
    'PPO + reflect':       game_reflect.logger.dir,
}

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
colors = ['steelblue', 'darkorange', 'green']

for (label, log_dir), color in zip(runs.items(), colors):
    scalars = load_tb_scalars(log_dir)
    for ax, tag, ylabel in [
        (axes[0], 'Train/Return Mean', 'Return'),
        (axes[1], 'Train/Success Rate', 'Success Rate'),
    ]:
        if tag in scalars:
            steps, vals = _unzip(scalars[tag])
            ax.plot(steps, _smooth(vals), label=label, color=color, linewidth=2)
            ax.plot(steps, vals, alpha=0.2, color=color)

for ax, title in zip(axes, ['Reward Curve', 'Success Rate']):
    ax.set_xlabel('Iteration'); ax.set_ylabel(title)
    ax.set_title(title); ax.legend(); ax.grid(True, alpha=0.3)

fig.suptitle('Condition Comparison', fontsize=13)
fig.tight_layout()
plt.show()


In [ ]:
# PPO loss components
fig = plot_ppo_losses(game_planner.logger.dir)
if fig:
    plt.show()


In [ ]:
# Reflection stats
fig = plot_reflection_stats(game_reflect.logger.dir)
if fig:
    plt.show()


## § 9  —  Reflection Memory Inspection

In [ ]:
# Build and inspect a reflection memory
mem = ReflectionMemory(maxlen=5, top_k=3, token_budget=300)

entries = [
    ('Exploring the perimeter found the key in fewer steps.',              True,  40, 1.0),
    ('Random exploration wasted 80 steps without finding the key.',        False, 128, 0.0),
    ('After picking up the key, going directly to the door succeeded.',    True,  35, 1.0),
    ('Revisiting explored regions provided no new information.',           False, 100, 0.0),
    ('Systematic boundary exploration improved key discovery rate.',       True,  38, 1.0),
    # 6th entry should evict oldest (maxlen=5)
    ('Holding the key and going straight to the door is most efficient.', True,  30, 1.0),
]

for i, (refl, success, ep_len, reward) in enumerate(entries):
    added = mem.add_memory(refl, episode_id=i, success=success, ep_len=ep_len, total_reward=reward)
    print(f'  ep {i}: added={added}')

print()
print(mem.summary())
print(f'Buffer size (max 5): {len(mem)}')

print('\n--- Context injected into planner (token-budget limited) ---')
print(mem.get_context())


In [ ]:
# Test deduplication
added = mem.add_memory('Exploring the perimeter found the key in fewer steps.', episode_id=99)
print(f'Duplicate added: {added}')   # Expected: False
print(f'Buffer size: {len(mem)}')    # Should be unchanged at 5

# Test success-only mode
mem_s = ReflectionMemory(maxlen=10, success_only=True)
mem_s.add_memory('Failure reflection', episode_id=0, success=False)
mem_s.add_memory('Success reflection', episode_id=1, success=True)
print(f'Success-only buffer size: {len(mem_s)} (expected 1)')
print(mem_s.summary())


In [ ]:
# Save / load round-trip
save_path = os.path.join(REPO_ROOT, 'log', 'test_memory_roundtrip.json')
mem.save(save_path)

mem2 = ReflectionMemory(maxlen=5, top_k=3)
mem2.load(save_path)
print(f'Reloaded: {len(mem2)} entries')
assert len(mem2) == len(mem)
print('Save/load round-trip OK.')


## § 10  —  Ablation Experiments

In [ ]:
# Ablation table utility
def run_ablation(conditions, n_itr=30, n_eval=5):
    """Run multiple conditions and return a comparison table."""
    results = []
    for label, kwargs in conditions:
        g = make_game(f'ablation-{label.replace(" ","-")}', n_itr=n_itr, **kwargs)
        g.train()
        returns, lengths, successes = [], [], []
        for _ in range(n_eval):
            ret, length, suc = g.evaluate(deterministic=True, record_frames=False)
            returns.append(ret); lengths.append(length); successes.append(suc)
        results.append({
            'label':        label,
            'mean_return':  round(float(np.mean(returns)),  3),
            'success_rate': round(float(np.mean(successes)), 3),
            'mean_ep_len':  round(float(np.mean(lengths)),  1),
        })
        print(results[-1])
    return results

ablation_conditions = [
    ('ppo-only',          dict(offline=True, reflection=False)),
    ('reflect-offline',   dict(offline=True, reflection=True, reflector_backend='offline')),
]

print('Running ablation (short runs)...')
ablation_results = run_ablation(ablation_conditions, n_itr=30)


In [ ]:
# Display ablation table
print('\n=== Ablation Results ===')
print(f'{"Condition":25s} | {"Mean Return":12s} | {"Success Rate":12s} | {"Mean Ep Len":12s}')
print('-' * 68)
for r in ablation_results:
    print(f"{r['label']:25s} | {r['mean_return']:12.3f} | {r['success_rate']:12.3f} | {r['mean_ep_len']:12.1f}")


## § 11  —  Failure Case Debugging

In [ ]:
# ── Inspect planner cache for invalid entities ────────────────────────────────
planner = game_planner.teacher_policy.planner
print(f'Plans cache entries: {len(planner.plans_dict)}')
print()

for obs_text, (plans, probs) in planner.plans_dict.items():
    print(f'OBS: {obs_text}')
    for plan, prob in zip(plans, probs):
        valid, errs = validate_plan(plan)
        status = '✅' if valid else f'❌ {errs}'
        print(f'  {prob:.2f} {status} → {plan}')
    print()


In [ ]:
# ── Planner stats (Bug 4 metrics) ─────────────────────────────────────────────
if hasattr(planner, 'planner_stats'):
    ps = planner.planner_stats
    print('=== Planner Stats ===')
    for k, v in ps.items():
        if isinstance(v, float):
            print(f'  {k:30s} = {v:.2%}')
        else:
            print(f'  {k:30s} = {v}')

    # Health check
    fallback_rate = ps.get('offline_fallback_rate', 0) + ps.get('explore_fallback_rate', 0)
    if fallback_rate > 0.5:
        print(f'\n⚠️ WARNING: High fallback rate {fallback_rate:.1%} — real Qwen inference may not be working')
    else:
        print(f'\n✅ Planner health OK — fallback rate {fallback_rate:.1%}')


In [ ]:
# ── Test hallucinated plan rejection ─────────────────────────────────────────
print('=== Hallucination / Invalid Plan Rejection Tests ===')
hallucinated_plans = [
    'go to <handle>, pick up <handle>',         # invalid object
    'move to (7, 3)',                            # coordinate
    'open <emergency exit>',                     # fabricated object
    'go to <hidden room>, search <bookshelves>', # hallucinated world
    'go to <key>, pick up <key>, go to (4,2)',   # mixed valid + coord
]
for plan in hallucinated_plans:
    result = parse_and_validate(plan)
    print(f'Input  : {plan}')
    print(f'Output : {result if result else "[REJECTED]"}')
    print()


In [ ]:
# ── Export experiment summary ────────────────────────────────────────────────
import json, datetime

summary = {
    'timestamp':   datetime.datetime.now().isoformat(),
    'config':      CFG,
    'ablation':    ablation_results,
    'parser_tests_passed': passed,
}
summary_path = os.path.join(REPO_ROOT, 'log', 'experiment_summary.json')
os.makedirs(os.path.dirname(summary_path), exist_ok=True)
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'Summary saved to {summary_path}')
